# SESSION 15: SQL + Python Integration

**Course:** Working with Databases & SQL Basics  
**Topic:** Integrating SQLite with Python using `sqlite3` and `pandas`  
**Database:** `foodie.db`  
**Execution Environment:** Python 3.12, SQLite 3.49, Pandas 3.1

---

## 📌 Session Overview & Objectives
In modern data science and analytics engineering, SQL and Python form a synergistic foundation. While SQL is the standard language for querying, aggregating, and extracting structured records from relational databases, Python—along with libraries like **Pandas**—provides machine learning capabilities, statistical manipulation, and programmatic workflow automation.

This session covers:
1. **The Python Standard Library `sqlite3`**: Establishing database connections and creating tables without external database servers.
2. **DML & Querying via Cursor**: Parameterized execution, committing transactions, and fetching filtered records.
3. **Pandas Integration**: Loading relational tables directly into DataFrames using `pd.read_sql_query()`.
4. **Data Transformation with Vectorized Operations & `.apply()`**: Adding scalar columns and computing row-wise conditional logic with lambda functions.
5. **Workflow Automation**: Scripting an automated daily summary pipeline that extracts high-value records and exports them to CSV format.

---

## Task 1: Installing / Importing `sqlite3` & Creating `foodie.db`
> **Task 1:** Install the `sqlite3` module in Python and write a script to create a new database called `foodie.db` with a table `Restaurants (id, name, cuisine, rating)`.

### Note on `sqlite3` Installation:
The `sqlite3` module is part of Python's **Standard Library** (PEP 249 Database API 2.0). It comes pre-installed with every standard Python distribution and requires **no external `pip install`**. Attempting to run `pip install sqlite3` in modern Python environments is redundant and will either fail or install an obsolete dummy package.

Let's verify our environment and create `foodie.db` with the `Restaurants` table schema:
- `id`: `INTEGER PRIMARY KEY`
- `name`: `TEXT NOT NULL`
- `cuisine`: `TEXT NOT NULL`
- `rating`: `REAL NOT NULL`

In [1]:
import os
import sqlite3
import pandas as pd

print(f"Python SQLite Module Version: {sqlite3.version}")
print(f"Underlying SQLite C-Library Version: {sqlite3.sqlite_version}")
print(f"Pandas Version: {pd.__version__}")

# Define database file path
db_filename = "foodie.db"

# Connect to database (creates file if not present)
conn = sqlite3.connect(db_filename)
cursor = conn.cursor()

# Create Restaurants table (idempotent setup)
cursor.execute("DROP TABLE IF EXISTS Restaurants;")
cursor.execute('''
CREATE TABLE Restaurants (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    cuisine TEXT NOT NULL,
    rating REAL NOT NULL
);
''')
conn.commit()

print("\n[OK] Database " + repr(db_filename) + " and table 'Restaurants' successfully initialized.")


Python SQLite Module Version: 2.6.0
Underlying SQLite C-Library Version: 3.49.1
Pandas Version: 3.1.0

[OK] Database 'foodie.db' and table 'Restaurants' successfully initialized.


C:\Users\Atharv\AppData\Local\Temp\ipykernel_22964\3916806449.py:5: DeprecationWarning: version is deprecated and will be removed in Python 3.14
  print(f"Python SQLite Module Version: {sqlite3.version}")


## Task 2: Inserting Sample Restaurants & Fetching Rating > 4.0
> **Task 2:** Using `sqlite3` in Python, insert three sample restaurants into the `Restaurants` table in `foodie.db` and write a query to fetch all restaurants with a rating above 4.0, then print their names.

We will insert 3 sample restaurants representing popular food categories:
1. **Little Italy** (Cuisine: *Italian*, Rating: *4.8*)
2. **Punjab Grill** (Cuisine: *North Indian*, Rating: *4.6*)
3. **Chai Point** (Cuisine: *Cafe*, Rating: *3.9*)

Notice that **Little Italy** and **Punjab Grill** have ratings $> 4.0$, whereas **Chai Point** ($3.9$) does not satisfy the `WHERE rating > 4.0` condition.

In [2]:
# Sample restaurants dataset
sample_restaurants = [
    (1, "Little Italy", "Italian", 4.8),
    (2, "Punjab Grill", "North Indian", 4.6),
    (3, "Chai Point", "Cafe", 3.9)
]

# Insert records using executemany for clean, parameterized SQL
cursor.executemany(
    "INSERT INTO Restaurants (id, name, cuisine, rating) VALUES (?, ?, ?, ?);",
    sample_restaurants
)
conn.commit()
print(f"[OK] Successfully inserted {cursor.rowcount} records into Restaurants table.")

# Execute query to fetch restaurants with rating > 4.0
query = "SELECT name FROM Restaurants WHERE rating > 4.0;"
cursor.execute(query)
high_rated_restaurants = cursor.fetchall()

print("\n--- Restaurants with Rating Above 4.0 ---")
for idx, (name,) in enumerate(high_rated_restaurants, start=1):
    print(f"{idx}. {name}")


[OK] Successfully inserted 3 records into Restaurants table.

--- Restaurants with Rating Above 4.0 ---
1. Little Italy
2. Punjab Grill


## Task 3: Loading Data into Pandas DataFrame & Displaying `head(2)`
> **Task 3:** Write Python code to load all rows from the `Restaurants` table in `foodie.db` into a Pandas DataFrame and display the top 2 rows using `DataFrame.head()`.

Pandas provides `pd.read_sql_query(sql, conn)` which directly executes SQL on a database connection and loads the results into a high-performance 2D DataFrame.

In [3]:
# Load all rows from Restaurants into a Pandas DataFrame
df_restaurants = pd.read_sql_query("SELECT * FROM Restaurants;", conn)

print("Full DataFrame Shape:", df_restaurants.shape)
print("\n--- Top 2 Rows using df_restaurants.head(2) ---")
df_restaurants.head(2)


Full DataFrame Shape: (3, 4)

--- Top 2 Rows using df_restaurants.head(2) ---


,id,name,cuisine,rating
0,1,Little Italy,Italian,4.8
1,2,Punjab Grill,North Indian,4.6


## Task 4: Adding `delivery_charge` & Computing Conditional `final_rating`
> **Task 4:** Add a new column `delivery_charge` to your DataFrame, setting it to 50 for all restaurants, and then calculate a new column `final_rating` as `rating + (0.1 if cuisine is 'Italian')`.  
> *Hint: Use `DataFrame.apply()` or a lambda function for the conditional logic.*

### Mathematical Formula:
$$\text{final\_rating} = \text{rating} + \begin{cases} 0.1 & \text{if cuisine} = \text{'Italian'} \\ 0.0 & \text{otherwise} \end{cases}$$

- **Little Italy**: $4.8 + 0.1 = 4.9$ (Bonus awarded)
- **Punjab Grill**: $4.6 + 0.0 = 4.6$ (No bonus)
- **Chai Point**: $3.9 + 0.0 = 3.9$ (No bonus)

In [4]:
# 1. Add scalar column delivery_charge
df_restaurants['delivery_charge'] = 50

# 2. Compute final_rating using DataFrame.apply with a lambda function
df_restaurants['final_rating'] = df_restaurants.apply(
    lambda row: round(row['rating'] + (0.1 if row['cuisine'] == 'Italian' else 0.0), 2),
    axis=1
)

print("--- Transformed Restaurants DataFrame ---")
display(df_restaurants)


--- Transformed Restaurants DataFrame ---


,id,name,cuisine,rating,delivery_charge,final_rating
0,1,Little Italy,Italian,4.8,50,4.9
1,2,Punjab Grill,North Indian,4.6,50,4.6
2,3,Chai Point,Cafe,3.9,50,3.9


## Task 5: Daily Summary Automation (Rating > 4.5 $\rightarrow$ CSV)
> **Task 5:** Automate a daily summary: Write a Python script that connects to `foodie.db`, fetches all restaurants with rating above 4.5, loads them into a DataFrame, and saves the result as a CSV file named `top_rated_restaurants.csv`.

This replicates an automated ELT / reporting workflow where a daily analytics cron job extracts high-performing restaurants and pushes them downstream to reporting dashboards or operational partners.

In [5]:
def export_daily_summary(db_name="foodie.db", output_csv="top_rated_restaurants.csv"):
    """Automated pipeline function to extract restaurants with rating > 4.5 into CSV."""
    with sqlite3.connect(db_name) as db_conn:
        query = """
        SELECT id, name, cuisine, rating
        FROM Restaurants
        WHERE rating > 4.5
        ORDER BY rating DESC;
        """
        df_top = pd.read_sql_query(query, db_conn)
    
    # Save to CSV without DataFrame row index
    df_top.to_csv(output_csv, index=False)
    print(f"[SUCCESS] Exported {len(df_top)} top-rated restaurants to '{output_csv}'.")
    return df_top

# Run automated summary
df_top_rated = export_daily_summary()

print("\n--- Content of top_rated_restaurants.csv ---")
display(df_top_rated)


[SUCCESS] Exported 2 top-rated restaurants to 'top_rated_restaurants.csv'.

--- Content of top_rated_restaurants.csv ---


,id,name,cuisine,rating
0,1,Little Italy,Italian,4.8
1,2,Punjab Grill,North Indian,4.6


In [6]:
# Verify CSV file by reading back with pd.read_csv
df_verified = pd.read_csv("top_rated_restaurants.csv")
print("Verified CSV Readback:")
display(df_verified)

# Close initial connection
conn.close()
print("\n[OK] Database connection closed cleanly.")


Verified CSV Readback:


,id,name,cuisine,rating
0,1,Little Italy,Italian,4.8
1,2,Punjab Grill,North Indian,4.6



[OK] Database connection closed cleanly.


---

## 🎯 Summary of Key Learnings & Production Best Practices

| Topic | Best Practice | Key Takeaway |
| :--- | :--- | :--- |
| **`sqlite3` Library** | Built-in standard library | Zero setup, serverless, atomic ACID-compliant transactions. |
| **Parameterized Queries** | Always use `?` placeholders | Prevents catastrophic SQL injection attacks. |
| **`pd.read_sql_query`** | Direct dataframe ingestion | Leverages fast C-level bindings directly from database cursor to columnar memory. |
| **Row-wise Logic** | `df.apply(..., axis=1)` | Perfect for multi-column conditional computations like feature engineering. |
| **Connection Safety** | Use `with sqlite3.connect(...)` | Guarantees automatic commit and connection closure even during runtime exceptions. |
